In [85]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [86]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/UGC-409_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/UGC-409_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     361   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        42   (2094, 964)   float32   


In [87]:
image_cut = image[55:175, 0:1890]
image_error_cut = image_error[55:175, 0:1890]

In [88]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.01736
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [89]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) #+
        #sersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [90]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/PSF/Real_seeing_UGC409.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 120, 1)

In [91]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(blue_area)
    print(green_area)
    print(purple_area)
    print(orange_area)
    print(choc_area)
    print('Sum',sum_area)
    print('Total', total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]     

## Halpha

In [92]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
sky_err = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Propag_errori/Sky_errors.txt")
real_errors_HA = np.sqrt(radial_profile_error_HA**2+(sky_err[2])**2+(0.04*radial_profile_HA)**2)

fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Fit/Halpha_fit.csv", index_col=0)

In [93]:
fit_HA['Component 5'].iloc[3] -3

99.8055

In [94]:
fit_HA['Component 5'].iloc[3] + 3

105.8055

In [95]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

3.3074247155937098
0.46384876863235946
8.241477526743463e-08
0.0032246747255889312
0.005483584512498776
Sum 3.7799818258789326
Total 3.7826303525191256
0.00044265144638590953
1.205136856515261
0.00012567074780566408
0.03599571991768693
0.012129606065281453
Sum 1.253830504692421
Total 1.2509383620197168
3.2839018169515803e-08
0.49304376086073126
1.3740173838459673
0.2516798148573682
0.016349924395079624
Sum 2.1350909167981644
Total 3.451102000007177
1.7048056201003677e-10
0.35562897407008914
0.016168972996352858
0.8388709691620722
0.02295677819720736
Sum 1.233625694596202
Total 1.2642625051881593
3.2932394162482623e-23
0.11067130844132275
1.3772875553078073e-08
0.0064599084666322505
0.4035910154804063
Sum 0.5207222461612369
Total 0.5215499841147861
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       87.498429   0.035304   0.000002  1.381947e-08  6.324369e-21
Green      12.271190  96.116409  23.092401  2.882795e+01  2.125342e+01
Magenta     0.000002   0.0100

In [96]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

5.720053912555967
0.9320525357585125
1.8856220082875046e-07
0.006594301063984199
0.010999045461918254
Sum 6.669699983402583
Total 6.670164552439788
0.0016033033373449425
2.3283266422130255
0.0005134517323651031
0.07503082812326767
0.02435687870888185
Sum 2.429831104114885
Total 2.4254461152991755
3.194577197404964e-07
1.3557362021406134
2.605082184889428
0.6880239662075858
0.04285979269348
Sum 4.691702465388826
Total 7.0759626997187235
1.8805739650023997e-09
0.9755057109163823
0.29228629567279046
1.8990769579600872
0.06013154571725049
Sum 3.2270005121470846
Total 3.551740676786291
2.1879013997750845e-22
0.222091405210949
3.055172183740291e-08
0.013257234861352812
0.7382144137434568
Sum 0.9735630843674804
Total 0.9745493222401136
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       85.761787   0.065984   0.000007  5.827622e-08  2.247313e-20
Green      13.974430  95.822571  28.896466  3.022949e+01  2.281223e+01
Magenta     0.000003   0.021131  55.525307  9.05

In [97]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

7.0831316832717475
1.4090622651210083
3.5149411843595504e-07
0.01026335429716503
0.01657860810596013
Sum 8.51903626229
Total 8.514708159558452
0.005186327540472448
3.3413108001552088
0.003860798026250829
0.12062304007880031
0.03678113893786896
Sum 3.507762104738602
Total 3.503392245013713
1.3034141117148976e-06
2.053334869766765
2.806933304833202
1.2076591653049973
0.06486145592474918
Sum 6.132790099243826
Total 8.679462468884447
8.45369180717787e-09
1.4749781110871463
0.9854139864335212
2.408981022738332
0.09117022531300525
Sum 4.960543354025697
Total 5.996361373244311
1.3993938029171968e-21
0.3350159455675991
5.4209475275248764e-08
0.020756384457551645
0.9949726645755707
Sum 1.3507450488101966
Total 1.3515744515038934
              Peak 1     Peak 2     Peak 3        Peak 4        Peak 5
Blue       83.144753   0.147853   0.000021  1.704187e-07  1.036016e-19
Green      16.540160  95.254772  33.481251  2.973420e+01  2.480231e+01
Magenta     0.000004   0.110064  45.769271  1.986504e+01 

## HBeta

In [98]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Fit/Hbeta_fit.csv", index_col = 0)

In [99]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.9575272379147258
0.2174380809956482
0.0
0.007843431793577691
5.129388978779204e-61
Sum 1.1828087507039515
Total 1.2248993956318455
0.043361558371983924
0.573680000930425
0.0005123489572715276
0.040040307363520836
5.282569012396877e-22
Sum 0.6575942156232013
Total 0.6572911586081536
0.011690808038123944
0.26987400247886956
10.962705730171479
0.14200355545360904
1.442891498709359e-10
Sum 11.386274096286371
Total 2.1173046765093653
0.007521473008321955
0.16917853771120525
0.00491033019846189
0.34254006879161725
8.083320557599706e-07
Sum 0.5241512180416621
Total 0.5032213900333402
0.0017923503634509057
0.005482007765716226
0.0
0.011928326653369608
0.16885980331668013
Sum 0.18806248809921688
Total 0.18811568676275275
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.095368e+01  6.593969e+00  1.026746e-01   1.434981   0.953061
Green      1.838320e+01  8.723921e+01  2.370170e+00  32.276666   2.914993
Magenta    0.000000e+00  7.791263e-02  9.628001e+01   

In [100]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.7996993782115271
0.4379535197248954
7.385153e-317
0.015852608352104433
7.725834718116649e-56
Sum 2.253505506288527
Total 2.333751991175611
0.08846668990571543
1.1311265654151796
0.027151080930288507
0.08177048213146215
9.080651147980749e-20
Sum 1.3285148183826456
Total 1.3053370148271126
0.03264733206126425
0.7414067951137648
30.793864694665675
0.37999085135850696
2.2881049227410834e-09
Sum 31.947909675487313
Total 5.902561023804807
0.02088171275361979
0.471638521244202
0.4305812869660743
0.8003475009615622
5.823928418989808e-06
Sum 1.7234548458538774
Total 1.3254035607065302
0.003600025945370881
0.011342122428563821
0.0
0.024148668202745933
0.33492921825315236
Sum 0.374020034829833
Total 0.3741457147476304
                  Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue        7.986221e+01  6.659067e+00  1.021893e-01   1.211619   0.962522
Green       1.943432e+01  8.514219e+01  2.320674e+00  27.365876   3.032491
Magenta    3.277184e-315  2.043717e+00  9.638773e+01  24

In [101]:
ddata1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


2.4100159080574386
0.6644683629088228
1.6050105694314224e-277
0.024199551780789898
5.2253492085840583e-51
Sum 3.0986838227470512
Total 3.1969576249055685
0.13724703197600183
1.6604171330782758
0.43058128696607423
0.12705701419767057
9.94637763980122e-18
Sum 2.3553024662180224
Total 1.9871653148773412
0.04989532517976757
1.1135939432918662
36.44776909149895
0.6284356716597452
2.8895136333786775e-08
Sum 38.239694060525466
Total 7.388124562215856
0.031793407273160434
0.7146880948385541
3.2392796189813593
1.0432718889266757
3.3350188585998045e-05
Sum 5.029066360208335
Total 2.2503421005719737
0.005438593735622025
0.01797427800289353
0.0
0.036966603434622276
0.494158257197462
Sum 0.5545377323705998
Total 0.5547601532219774
                  Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue        7.986221e+01  5.827151e+00  1.304805e-01   0.632193   0.980744
Green       1.943432e+01  7.049698e+01  2.912141e+00  14.211149   3.241308
Magenta    3.277184e-315  1.828136e+01  9.53139

## NII

In [102]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Fit/NII_fit.csv", index_col = 0)

In [103]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.074928235149077
0.12287588918750664
1.1861426401042425e-09
6.22980986743392e-06
0.021886629208097697
Sum 1.2196969845406915
Total 1.2492356260210284
0.12117072537722667
0.6257920019073409
1.1565071124960087e-05
0.0006529797246024101
0.039205858279956526
Sum 0.7868331303602515
Total 0.7867759507989535
0.03995335458341423
0.43306958750618263
0.947334815550996
0.03022416881016875
0.04372995250622573
Sum 1.4943118789569874
Total 3.239429067682745
0.028077388438571514
0.29464398574833617
0.008587037047065035
0.22031878946992356
0.05424613681423201
Sum 0.6058733375181282
Total 0.6337006320091814
0.009430134477494457
0.0013540386284700498
1.2970565729684284e-10
2.616256411303873e-05
0.23727697785567098
Sum 0.2480873136554542
Total 0.2485407501560687
                 Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       8.813076e+01  15.399800   2.673696   4.634201  3.801135e+00
Green      1.007430e+01  79.533001  28.981205  48.631284  5.457912e-01
Magenta    9.724896e-08   0.0014

In [104]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.9228183488319823
0.2516956585800435
2.895615033254881e-09
1.3496964796787312e-05
0.04384281744459315
Sum 2.2183703247170308
Total 2.2606702841098527
0.24576419748055517
1.2412620690818152
8.902678343613408e-05
0.0014926338784838726
0.07855634302091044
Sum 1.5671642702452009
Total 1.567090553751207
0.11037256914536595
1.1661316752112172
1.7866168063634096
0.09397587651637107
0.11514070437197514
Sum 3.2722376316083386
Total 6.452249618686779
0.07722116193027459
0.8139041310756197
0.18710855855503622
0.46549078542514183
0.1427851391101741
Sum 1.6865097760962464
Total 2.0855575869971696
0.018913886923588318
0.003044027963534419
3.0261367301406046e-10
5.731350765296869e-05
0.4611464285088329
Sum 0.4831616572062223
Total 0.48380572426407575
                 Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       8.667707e+01  15.682095   3.373000   4.578756  3.914608e+00
Green      1.134597e+01  79.204337  35.637133  48.259675  6.300227e-01
Magenta    1.305289e-07   0.005681  54.5

In [105]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

2.524504227715958
0.39208873446966863
5.952193346598808e-09
2.3044619851862287e-05
0.06593846819863433
Sum 2.9825544809563063
Total 3.0266407451509343
0.3775384508880882
1.8367545351061434
0.0017097271925939337
0.0027791622754175446
0.11819704709723941
Sum 2.3369789225594824
Total 2.337966732975183
0.16766501502306486
1.728265724059189
1.916175439145098
0.20309535210131668
0.17331672540715723
Sum 4.188518255735826
Total 7.583707470091934
0.1169960316616034
1.219681163601017
0.6583776747285179
0.555659999699941
0.2150016270681936
Sum 2.765716496759273
Total 4.0923354607085
0.02850550917592975
0.005456748534624476
5.801034405465597e-10
9.960609266448262e-05
0.6666834261885561
Sum 0.7007452905718783
Total 0.7014093143353
                 Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       8.464235e+01  16.154979   4.002967   4.230225  4.067885e+00
Green      1.314607e+01  78.595255  41.261984  44.100007  7.787064e-01
Magenta    1.995670e-07   0.073160  45.748289  23.804959  8

## SII

In [106]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Fit/SII_fit.csv", index_col = 0)

In [107]:
fit

,Component 1,Component 2,Component 3,Component 4,Component 5
I_e,1.4034,0.1438,3.0893,0.0061,0.0270
r_e,0.0493,20.0000,0.0039,19.0727,19.9962
n,3.7732,0.2650,2.8817,1.8462,0.5467
x_0,27.4477,45.4205,61.3599,68.6207,100.2260


In [116]:
def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(blue_area)
    print(green_area)
    print(purple_area)
    print(orange_area)
    print(choc_area)
    print('Sum',sum_area)
    print('Total', total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]    

In [117]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.2883813719912773
0.2644387027864846
1.1333394883399191e-05
0.009817926279719912
9.500068467925922e-51
Sum 1.5626493344523653
Total 1.5065185175933058
0.07321645153380968
0.5770382219534149
0.00269696309208343
0.07574537572422771
1.0573144212682134e-19
Sum 0.7286970123035357
Total 0.7287481152179461
0.021974669001353905
0.2718053069779001
2.2962219602123453
0.21475510679359627
7.904024366570163e-10
Sum 2.804757043775598
Total 2.3444559060429833
0.014823636532717392
0.17023034629242303
0.0051043841292792045
0.3357492767525228
1.932884463750144e-06
Sum 0.5259095765914061
Total 0.5251542467267971
0.0042714304208826825
0.003903176569136928
1.7804542292844803e-06
0.017560535262655934
0.2194550219977939
Sum 0.24519194470469874
Total 0.24526804415229037
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.244853e+01  1.004758e+01  7.834785e-01   2.818666   1.742076
Green      1.692246e+01  7.918768e+01  9.690868e+00  32.368748   1.591886
Magenta    7.252680e

In [118]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.3785444375563998
0.5312794107105565
2.4274956865212734e-05
0.02000639041121538
9.565913163132123e-47
Sum 2.929854513635037
Total 2.8315787090120885
0.14887772572039967
1.1421162699549492
0.013907795648847053
0.15443383463224963
8.045834856250423e-18
Sum 1.4593356259564456
Total 1.457420044790526
0.06100736741024822
0.745662510264633
6.3925923486336
0.5561173912798925
9.571631136941939e-09
Sum 7.755379627160005
Total 6.37101569802309
0.040944048521408316
0.4746487706764261
0.11635612693650278
0.8416441907235668
1.2054702668381393e-05
Sum 1.4736051915605721
Total 1.4554034860025928
0.008572238867289517
0.008156296110918457
3.7181429532797223e-06
0.035787856049669976
0.4348058336884282
Sum 0.4873259428592594
Total 0.48750187730877304
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.118302e+01  1.020175e+01  7.866458e-01   2.778495   1.759036
Green      1.813330e+01  7.826276e+01  9.614778e+00  32.210037   1.673684
Magenta    8.285380e-04  9.530224e-

In [119]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

3.13991411586001
0.8027283749938409
4.0802137094430555e-05
0.030950222054668557
5.464232244920297e-43
Sum 3.973633515045614
Total 3.855961806132944
0.22967004744592295
1.685122432075251
0.11251921885538928
0.23912977475483405
4.3732128454106206e-16
Sum 2.266441473131398
Total 2.2385835026482286
0.09292606691743803
1.118673325480897
7.586755996383992
0.8514489556663039
9.469887271447637e-08
Sum 9.649804439147506
Total 7.963524357058971
0.06216889058273352
0.7187185221842525
0.7311550274196126
1.1728973733028702
6.04331264548412e-05
Sum 2.6850002466159233
Total 2.5541685174164455
0.01293220377943181
0.013127808774133182
5.991531056354073e-06
0.05537474889862927
0.64035804319756
Sum 0.7217987961808106
Total 0.7221018024317265
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       7.901871e+01  1.013351e+01  9.629839e-01   2.315415   1.791663
Green      2.020137e+01  7.435102e+01  1.159270e+01  26.767913   1.818763
Magenta    1.026822e-03  4.964576e+00  7.8620

## OIII

In [112]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Fit/OIII_fit.csv", index_col = 0)

In [113]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.2883813719912773
0.2644387027864846
1.1333394883399191e-05
0.009817926279719912
9.500068467925922e-51
Sum 1.5626493344523653
Total 1.5065185175933058
0.07321645153380968
0.5770382219534149
0.00269696309208343
0.07574537572422771
1.0573144212682134e-19
Sum 0.7286970123035357
Total 0.7287481152179461
0.021974669001353905
0.2718053069779001
2.2962219602123453
0.21475510679359627
7.904024366570163e-10
Sum 2.804757043775598
Total 2.3444559060429833
0.014823636532717392
0.17023034629242303
0.0051043841292792045
0.3357492767525228
1.932884463750144e-06
Sum 0.5259095765914061
Total 0.5251542467267971
0.0042714304208826825
0.003903176569136928
1.7804542292844803e-06
0.017560535262655934
0.2194550219977939
Sum 0.24519194470469874
Total 0.24526804415229037
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.244853e+01  1.004758e+01  7.834785e-01   2.818666   1.742076
Green      1.692246e+01  7.918768e+01  9.690868e+00  32.368748   1.591886
Magenta    7.252680e

In [114]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.3785444375563998
0.5312794107105565
2.4274956865212734e-05
0.02000639041121538
9.565913163132123e-47
Sum 2.929854513635037
Total 2.8315787090120885
0.14887772572039967
1.1421162699549492
0.013907795648847053
0.15443383463224963
8.045834856250423e-18
Sum 1.4593356259564456
Total 1.457420044790526
0.06100736741024822
0.745662510264633
6.3925923486336
0.5561173912798925
9.571631136941939e-09
Sum 7.755379627160005
Total 6.37101569802309
0.040944048521408316
0.4746487706764261
0.11635612693650278
0.8416441907235668
1.2054702668381393e-05
Sum 1.4736051915605721
Total 1.4554034860025928
0.008572238867289517
0.008156296110918457
3.7181429532797223e-06
0.035787856049669976
0.4348058336884282
Sum 0.4873259428592594
Total 0.48750187730877304
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.118302e+01  1.020175e+01  7.866458e-01   2.778495   1.759036
Green      1.813330e+01  7.826276e+01  9.614778e+00  32.210037   1.673684
Magenta    8.285380e-04  9.530224e-

In [115]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

3.13991411586001
0.8027283749938409
4.0802137094430555e-05
0.030950222054668557
5.464232244920297e-43
Sum 3.973633515045614
Total 3.855961806132944
0.22967004744592295
1.685122432075251
0.11251921885538928
0.23912977475483405
4.3732128454106206e-16
Sum 2.266441473131398
Total 2.2385835026482286
0.09292606691743803
1.118673325480897
7.586755996383992
0.8514489556663039
9.469887271447637e-08
Sum 9.649804439147506
Total 7.963524357058971
0.06216889058273352
0.7187185221842525
0.7311550274196126
1.1728973733028702
6.04331264548412e-05
Sum 2.6850002466159233
Total 2.5541685174164455
0.01293220377943181
0.013127808774133182
5.991531056354073e-06
0.05537474889862927
0.64035804319756
Sum 0.7217987961808106
Total 0.7221018024317265
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       7.901871e+01  1.013351e+01  9.629839e-01   2.315415   1.791663
Green      2.020137e+01  7.435102e+01  1.159270e+01  26.767913   1.818763
Magenta    1.026822e-03  4.964576e+00  7.8620